# UC Persistent Admission-Rate Gaps

This reproducible Colab analysis answers the dashboard question before the Streamlit presentation layer: among represented California public-high-school applicants, which high-school-site and UC-campus combinations show persistent applicant-weighted actual-minus-provided-expected admission-rate gaps during 2017–2025?

The analysis excludes 2022 because the supplied expected-rate baseline is unavailable. It preserves blanks as unknown, excludes `Universitywide` from campus comparisons, and makes descriptive—not causal—claims. Upload `Data/dashboard_data.csv` to Colab, or place it beside this notebook when running from the repository.

In [ ]:
from pathlib import Path

import altair as alt
import pandas as pd

RESIDUAL_YEARS = (2017, 2018, 2019, 2020, 2021, 2023, 2024, 2025)
DATA_CANDIDATES = (Path("Data/dashboard_data.csv"), Path("dashboard_data.csv"))
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Upload Data/dashboard_data.csv or dashboard_data.csv to Colab before running this cell.")

df = pd.read_csv(DATA_PATH, low_memory=False)
required = {"fall_term", "campus", "atp_code", "high_school", "city", "applicants", "admits", "expected_admit_rate"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
print(f"Loaded {len(df):,} rows from {DATA_PATH}")
print(df.head().to_string(index=False))

## 1. Build residual-ready rows

Admission rates are calculated from pooled counts. The provided expected rate is weighted by applicants when rows are aggregated. No missing value is converted to zero.

In [ ]:
eligible = df.copy()
eligible["fall_term"] = pd.to_numeric(eligible["fall_term"], errors="coerce")
for column in ("applicants", "admits", "expected_admit_rate"):
    eligible[column] = pd.to_numeric(eligible[column], errors="coerce")
eligible = eligible[
    eligible["fall_term"].isin(RESIDUAL_YEARS)
    & eligible["campus"].notna()
    & eligible["atp_code"].notna()
    & eligible["applicants"].notna()
    & eligible["admits"].notna()
    & eligible["expected_admit_rate"].notna()
    & (eligible["applicants"] > 0)
    & (eligible["campus"] != "Universitywide")
].copy()
eligible["actual_rate"] = eligible["admits"] / eligible["applicants"]
eligible["expected_rate"] = eligible["expected_admit_rate"]
eligible["residual"] = eligible["actual_rate"] - eligible["expected_rate"]
print(f"Residual-ready rows: {len(eligible):,}")
print("Residual years: " + ", ".join(map(str, RESIDUAL_YEARS)) + "; 2022: baseline unavailable")

## 2. Identify persistent school-site/campus combinations

Persistence requires at least three residual years, at least 80% of observed yearly residuals on one side of zero, and agreement between the pooled residual sign and the dominant yearly direction.

In [ ]:
records = []
for (atp_code, campus), group in eligible.groupby(["atp_code", "campus"], sort=False):
    if len(group) < 3:
        continue
    positive = int((group["residual"] > 0).sum())
    negative = int((group["residual"] < 0).sum())
    if positive == negative:
        continue
    direction = "positive" if positive > negative else "negative"
    consistency = max(positive, negative) / len(group)
    if consistency < 0.80:
        continue
    applicants = group["applicants"].sum()
    admits = group["admits"].sum()
    actual_rate = admits / applicants
    expected_rate = (group["applicants"] * group["expected_rate"]).sum() / applicants
    pooled_residual = actual_rate - expected_rate
    if (direction == "positive" and pooled_residual <= 0) or (direction == "negative" and pooled_residual >= 0):
        continue
    first = group.iloc[0]
    records.append({
        "atp_code": atp_code, "campus": campus, "high_school": first["high_school"], "city": first["city"],
        "pooled_applicants": applicants, "pooled_admits": admits, "actual_rate": actual_rate,
        "expected_rate": expected_rate, "pooled_residual": pooled_residual, "direction": direction,
        "years_observed": len(group), "direction_consistency": consistency,
        "limited_evidence": len(group) < 5 or applicants < 100,
    })
gaps = pd.DataFrame(records).sort_values(["direction", "pooled_residual"], ascending=[True, False], ignore_index=True)
print(f"Persistent combinations: {len(gaps)}")
print(gaps["direction"].value_counts().to_dict())
print(gaps.head().to_string(index=False))

## 3. Primary visual: zero-centered diverging ranking

The chart shows the ten largest positive and ten largest negative pooled gaps. Position relative to the zero line and the text labels communicate direction; color is supplemental.

In [ ]:
positive = gaps[gaps["direction"] == "positive"].nlargest(10, "pooled_residual")
negative = gaps[gaps["direction"] == "negative"].nsmallest(10, "pooled_residual")
ranking = pd.concat([negative, positive], ignore_index=True).copy()
ranking["label"] = ranking["high_school"].fillna("Unknown") + " · " + ranking["city"].fillna("Unknown") + " — " + ranking["campus"]
ranking["residual_pp"] = ranking["pooled_residual"] * 100
ranking = ranking.sort_values("residual_pp")
max_abs = max(abs(ranking["residual_pp"]).max(), 1)
bars = alt.Chart(ranking).mark_bar().encode(
    y=alt.Y("label:N", sort="-x", title="High-school site · city · campus"),
    x=alt.X("residual_pp:Q", title="Actual minus provided expected rate (percentage points)", scale=alt.Scale(domain=[-max_abs, max_abs])),
    color=alt.Color("direction:N", title="Direction", scale=alt.Scale(domain=["negative", "positive"], range=["#c2410c", "#0369a1"])),
    tooltip=[alt.Tooltip("label:N", title="School/campus"), alt.Tooltip("residual_pp:Q", title="Residual (pp)", format=".2f"), alt.Tooltip("pooled_applicants:Q", title="Pooled applicants"), alt.Tooltip("years_observed:Q", title="Years observed")],
)
zero = alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#172033", strokeWidth=1.5).encode(x="zero:Q")
(bars + zero).properties(title="Persistent school-campus gaps", height=max(300, len(ranking) * 28))

## 4. Supporting visual: applicant-weighted campus/year context

In [ ]:
eligible["expected_admits"] = eligible["applicants"] * eligible["expected_rate"]
context = eligible.groupby(["fall_term", "campus"], as_index=False).agg(
    applicants=("applicants", "sum"), admits=("admits", "sum"), expected_admits=("expected_admits", "sum")
)
context["actual_rate"] = context["admits"] / context["applicants"]
context["expected_rate"] = context["expected_admits"] / context["applicants"]
context["residual_pp"] = (context["actual_rate"] - context["expected_rate"]) * 100
lines = alt.Chart(context).mark_line(point=True).encode(
    x=alt.X("fall_term:O", title="Fall year"),
    y=alt.Y("residual_pp:Q", title="Residual (percentage points)"),
    color=alt.Color("campus:N", title="Campus"),
    tooltip=[alt.Tooltip("fall_term:O", title="Fall year"), alt.Tooltip("campus:N", title="Campus"), alt.Tooltip("residual_pp:Q", title="Residual (pp)", format=".2f")],
)
zero = alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#172033", strokeWidth=1.2).encode(y="zero:Q")
(lines + zero).properties(title="Applicant-weighted campus context", height=360)

## Interpretation boundary

The results are descriptive patterns in aggregated school-level data. They do not establish causality, fairness, institutional intent, or individual admission odds. Low-volume or incomplete combinations remain visible but are labeled `limited_evidence` in the result table.